# MAS

In [24]:
from src.board.core import Board

board = Board()
tools = board.tools
tools

[StructuredTool(name='get_tasks', description='Возвращает перечень доступных задач с фильтром:\n- status : Status | None - только заданный статус (если None, фильтр не применяется). По умолчанию None.\n- priority : Priority - заданный приоритет и выше (по умолчанию Priority.TRIVIAL) \nЕсли фильтр не указан, вернется весь перечень задач.', args_schema=<class 'langchain_core.utils.pydantic.get_tasks'>, func=<bound method Board.get_tasks of Board(iteration=0, tasks=[], results=[])>),
 StructuredTool(name='update_status', description='Обновляет текущий статус задачи.\nВозвращает None, если задача с таким ID не найдена.\nВозвращает текущее состояние задачи.\nВНИМАНИЕ: сверяйте желаемое и текущее состояние задачи.', args_schema=<class 'langchain_core.utils.pydantic.update_status'>, func=<bound method Board.update_status of Board(iteration=0, tasks=[], results=[])>),
 StructuredTool(name='update_priority', description='Обновляет текущий приоритет задачи.\nВозвращает None, если задача с таким 

In [25]:
from langgraph.graph import MessagesState

class State(MessagesState):
    last_messages_len : int
    manager_response : str

In [26]:
from src import Agent
from langchain.messages import AIMessage


manager_tools = [tool for tool in tools if tool.name not in ['add_result']]

manager_prompt = """
Ты менеджер. 

ТВОИ ОБЯЗАННОСТИ:
- Декомпозиция запроса пользователя и создание задач для агентов.
- Отслеживание прогресса по задачам.
- Суммаризация результатов и формирование ответа пользователю.

ТЫ ТАКЖЕ МОЖЕШЬ:
- Помечать задачи как DONE или CANCELLED.
- Менять приоритет задач. 

ВАЖНО:
- Работа заканчивается тогда, когда все задачи на доске либо CANCELLED, либо DONE.
- Если ты готов отдать ответ пользователю, убедись, что все задачи имеют необходимый статус для завершения.
- Агенты ничего не знают про запрос пользователя. При постановке задачи убедись, что информации достаточно.

ТЫ НЕ ДОЛЖЕН отвечать на основе своих знаний. Отвечай только на основе результатов по задачам.
"""

manager_agent = Agent(tools=manager_tools, system_prompt=manager_prompt)

def manager_node(state : State):
    board.update_iteration()
    
    messages = state['messages']
    last_messages_len = state.get('last_messages_len',0)
    new_messages = messages[last_messages_len:]

    result = manager_agent.run(new_messages)

    return {
        'manager_response': result,
        'last_messages_len': len(messages)  # обновляем индекс
    }

In [27]:
def loop_node(state : State):
    return {}

In [ ]:
agent_prompt = """
Ты {role}. Твои обязанности:
- Проверка пула новых задач.
- Выполнение задач, относящихся к твоей специализации.

ПРАВИЛА РАБОТЫ:
- Если можешь, выполняй задачу без лишних слов.
- Если чего-то не хватает, напиши.
- Если не относится к тебе - ничего не делай.
- Ты не можешь помечать задачи как CANCELLED или DONE.
"""

agent_tools = [tool for tool in tools if tool.name not in ['update_priority', 'add_tasks']]

agents = {role:Agent(
    tools=agent_tools, 
    system_prompt=agent_prompt.format(role=role)    
) for role in [
    'специалист по стратегическому планированию',
    'специалист по территориальному планированию',
    'специалист по урбанистике',
]}

def create_node(agent):

    def agent_node(state : State):
        message = AIMessage(state['manager_response'])
        result = agent.run([message])
        return {'messages': [AIMessage(result)]}     
    
    return agent_node    

agents_nodes = {role: create_node(agent) for role,agent in agents.items()}

In [29]:
from langgraph.graph import StateGraph, START, END
from src.board.models import Status

graph = StateGraph(State)

graph.add_node('manager', manager_node)
graph.add_node('loop', loop_node)

def loop_condition(state : State):
    tasks = board.get_tasks()
    finish = all([t.status in [Status.CANCELLED, Status.DONE] for t in tasks])
    if finish:
        return END
    return 'loop'
        
graph.add_edge(START, 'manager')
graph.add_conditional_edges('manager', loop_condition, [END, 'loop'])

for role,node in agents_nodes.items():
    graph.add_node(role, node)
    graph.add_edge('loop', role)
    graph.add_edge(role, 'manager')


app = graph.compile(debug=True)

In [ ]:
app.invoke({'messages': ['Как должен развиваться г. Гатчина (Ленинградская область)? Мне нужен примерный ответ']})

[values] {'messages': [HumanMessage(content='Как должен развиваться г. Гатчина (Ленинградская область)? Мне нужен примерный ответ', additional_kwargs={}, response_metadata={}, id='a6e6fe72-eea2-434e-834d-ba7d7afc5628')]}
[updates] {'manager': {'manager_response': 'Сейчас я собираю необходимые материалы\u202f— официальные стратегии развития, сведения о текущих инфраструктурных проектах и прогнозы демографического и экономического роста Гатчины. Как только все задачи будут завершены и результаты получены, я подготовлю для вас подробный примерный план развития города. Спасибо за терпение!', 'last_messages_len': 1}}
[values] {'messages': [HumanMessage(content='Как должен развиваться г. Гатчина (Ленинградская область)? Мне нужен примерный ответ', additional_kwargs={}, response_metadata={}, id='a6e6fe72-eea2-434e-834d-ba7d7afc5628')], 'last_messages_len': 1, 'manager_response': 'Сейчас я собираю необходимые материалы\u202f— официальные стратегии развития, сведения о текущих инфраструктурных 

In [ ]:
board

Board(iteration=1, tasks=[Task(name='Сбор данных о Гатчине', description='Собрать актуальные данные о социально-экономическом положении города Гатчина: население, экономика, инфраструктура, туризм, бюджет.', priority=<Priority.MEDIUM: 3>, id='bb39830062a04fd1abd3f61961ec43ff', status=<Status.TODO: 'todo'>, created_at=1), Task(name='Анализ проблем и возможностей', description='Проанализировать основные проблемы и возможности развития Гатчины: транспорт, жильё, промышленность, культурное наследие, экология.', priority=<Priority.MEDIUM: 3>, id='b18f8101bd764272abc9be3faa9fbb58', status=<Status.TODO: 'todo'>, created_at=1), Task(name='Разработка плана развития', description='Сформировать примерный быстрый план развития города с приоритетными направлениями и конкретными рекомендациями (инфраструктура, бизнес, туризм, цифровизация).', priority=<Priority.MEDIUM: 3>, id='5294a606c30e46c59ddd286b3c4f2ef9', status=<Status.TODO: 'todo'>, created_at=1), Task(name='Подготовка итогового ответа', des